# Cross-check: pymannkendall and the analytic ITA significance limits

**Run this on your own machine** (it needs `pymannkendall`; first run `pip install pymannkendall`, then restart the kernel).

What it does
1. Rebuilds the 32 corrected SPI/SPEI series (same cells as the main notebook, no bootstrap needed for this step).
2. Runs the Mann-Kendall tests from **pymannkendall** (`original_test`, `hamed_rao_modification_test`) and compares them with my own implementation, so you can confirm my code gives the same p-values, Z and Sen slopes.
3. Computes the **analytic ITA significance limits of Sen (2017)** and compares them with the block-bootstrap interval and with modified Mann-Kendall.
4. Computes the **Alashan (2020)** ITA test (Z-statistic with unequal-variance standard deviation of the slope).

**About the Sen (2017) formula.** The function uses the formula as given in the text you supplied: sigma_s^2 = 8 sigma^2 (1 - rho_12) / n^2, with rho_12 the correlation between the two **sorted** halves. In simulations (independent data) it rejects about 4% of the time under no change, so it behaves like a proper 5% test; it is *not* valid for autocorrelated series (about 25% false positives at AR(1) = 0.6 and 47% at 0.9). Please still confirm the equation against the paper itself.
- Sen, Z. (2017). Innovative trend significance test and applications. *Theoretical and Applied Climatology*.
- Alashan, S. (2020). Combination of modified Mann-Kendall method and Sen innovative trend analysis. *Engineering Reports*.

**Alashan (2020) as implemented:** sigma_s = (2/n) * sqrt(s1^2 + s2^2 - 2 rho12 s1 s2) using the standard deviations of the two sorted halves, and Z_ITA = s / sigma_s, compared with +/-1.96. Like Sen (2017), it assumes independent observations.

In [11]:
import os, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display
from math import sin, radians, acos, tan, pi
from scipy import stats

# ---- EDIT THESE TWO LINES IF NEEDED -------------------------------------------
DATA = 'reshaped_climate_data.xlsx'   # the workbook your first notebook created (same folder as this notebook)
OUT  = 'sh_ita_outputs'               # folder where tables and figures will be saved
# ------------------------------------------------------------------------------
os.makedirs(OUT, exist_ok=True)
%matplotlib inline

## Settings

In [12]:
LAT = {'YENAGOA': 4.95, 'IKOT AKPADEN': 4.62, 'IBADAN': 7.38, 'ABUJA': 9.07,
       'JALINGO': 8.89, 'SOKOTO': 13.01, 'GOMBE': 10.28, 'KUKAWA': 12.92}
X_START, X_END = '1982-01-01', '2001-12-31'      # first half  (240 months)
Y_START, Y_END = '2002-01-01', '2021-12-31'      # second half (240 months)
B = 2000                                          # bootstrap replicates
BLOCK = {3: 12, 12: 24}                           # block length (months) by timescale
MIN_N = 10                                        # min class size for a trend label
SEED = 2026
CLIP = 1e-6
COL = {'Low': 'red', 'Medium': 'orange', 'High': 'blue'}   # ONE colour convention for every figure

## Load data

In [13]:
P = pd.read_excel(DATA, 'Precip_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
T = pd.read_excel(DATA, 'Temp_Wide', index_col='Date').drop(columns=['Year', 'Month'], errors='ignore')
P.index = pd.to_datetime(P.index); T.index = pd.to_datetime(T.index)

## PET (Thornthwaite)

In [14]:
MID = [15, 46, 75, 105, 135, 166, 196, 227, 258, 288, 319, 349]
def daylength(lat, month):
    decl = radians(23.45 * sin(radians(360 * (284 + MID[month - 1]) / 365.0)))
    x = max(min(-tan(radians(lat)) * tan(decl), 0.999999), -0.999999)
    return 24.0 / pi * acos(x)
def thornthwaite(t, lat):
    clim = t.groupby(t.index.month).mean()
    I = ((clim.clip(lower=0) / 5.0) ** 1.514).sum()
    a = 6.75e-7 * I ** 3 - 7.71e-5 * I ** 2 + 1.792e-2 * I + 0.49239
    out = [16.0 * (10.0 * v / I) ** a * (daylength(lat, d.month) / 12.0) * (d.days_in_month / 30.0)
           if v > 0 else 0.0 for d, v in t.items()]
    return pd.Series(out, index=t.index)
PET = pd.DataFrame({l: thornthwaite(T[l], LAT[l]) for l in P.columns})

## SPI and SPEI per calendar month

In [15]:
def roll(s, sc): return s.rolling(sc, min_periods=sc).sum()
def spi_cm(p, sc):
    a = roll(p, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); nz = v[v > 0]
        if len(nz) < 10: continue
        q = (v == 0).mean()
        g, _, sc_ = stats.gamma.fit(nz.values, floc=0)
        cdf = np.where(v.values <= 0, q, q + (1 - q) * stats.gamma.cdf(v.values, g, loc=0, scale=sc_))
        out[v.index] = stats.norm.ppf(np.clip(cdf, CLIP, 1 - CLIP))
    return out
def spei_cm(p, pet, sc):
    a = roll(p - pet, sc); out = pd.Series(np.nan, index=a.index)
    for m in range(1, 13):
        v = a[a.index.month == m].dropna(); N = len(v)
        if N < 20: continue
        out[v.index] = stats.norm.ppf((stats.rankdata(v.values) - 0.44) / (N + 0.12))
    return out

In [16]:
def cbb(a, L, rng):          # circular moving-block bootstrap (same as in the main notebook)
    n = len(a); nb = int(np.ceil(n / L))
    return a[(rng.integers(0, n, nb)[:, None] + np.arange(L)).ravel()[:n] % n]

## My Mann-Kendall implementation (same as in the benchmark notebook)

In [17]:
def sen_slope(x):
    x = np.asarray(x, float); n = len(x)
    return np.median(np.concatenate([(x[k:] - x[:-k]) / k for k in range(1, n)]))

def mk_test(x, hr=True):
    """Mann-Kendall S, Z, p-value (two-sided) and Sen slope per time step. hr=True applies Hamed & Rao (1998)."""
    x = np.asarray(x, float); n = len(x)
    S = sum(np.sign(x[i + 1:] - x[i]).sum() for i in range(n - 1))
    _, cnt = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(cnt * (cnt - 1) * (2 * cnt + 5))) / 18.0
    beta = sen_slope(x); f = 1.0
    if hr:
        r = stats.rankdata(x - beta * np.arange(n)); r = r - r.mean(); den = (r * r).sum()
        lim = stats.norm.ppf(0.975) / np.sqrt(n); sni = 0.0
        for k in range(1, n - 1):
            a = (r[:-k] * r[k:]).sum() / den
            if abs(a) > lim: sni += (n - k) * (n - k - 1) * (n - k - 2) * a
        f = 1 + 2.0 / (n * (n - 1) * (n - 2)) * sni
        f = f if f > 0 else 1.0
    z = 0.0 if S == 0 else (S - np.sign(S)) / np.sqrt(var * f)
    return dict(S=S, Z=z, p=2 * (1 - stats.norm.cdf(abs(z))), sen=beta, vcf=f)

def direction(p, est, alpha=0.05):
    return '0' if not (p < alpha) else ('+' if est > 0 else '-')

## pymannkendall versus my implementation

In [18]:
try:
    import pymannkendall as mk
except ImportError:
    raise ImportError("Install it first:  pip install pymannkendall   (then restart the kernel)")

X_S, Y_E = X_START, Y_END
def series_for(loc, kind, sc):
    s = spi_cm(P[loc], sc) if kind == 'SPI' else spei_cm(P[loc], PET[loc], sc)
    return s.loc[X_S:Y_E].dropna()

rows = []
for sc in (3, 12):
    for kind in ('SPI', 'SPEI'):
        for loc in P.columns:
            v = series_for(loc, kind, sc).values
            hr, og = mk.hamed_rao_modification_test(v), mk.original_test(v)
            mine, mine0 = mk_test(v, hr=True), mk_test(v, hr=False)
            rows.append(dict(Location=loc.title(), Index=f'{kind}-{sc}',
                             p_pkg_HR=hr.p, p_mine_HR=mine['p'], z_pkg_HR=hr.z, z_mine_HR=mine['Z'],
                             sen_pkg_per_decade=hr.slope * 120, sen_mine_per_decade=mine['sen'] * 120,
                             p_pkg_orig=og.p, p_mine_orig=mine0['p']))
CK = pd.DataFrame(rows)
print('max |p difference|  (Hamed-Rao):', (CK.p_pkg_HR - CK.p_mine_HR).abs().max())
print('max |Z difference|  (Hamed-Rao):', (CK.z_pkg_HR - CK.z_mine_HR).abs().max())
print('max |Sen slope difference|     :', (CK.sen_pkg_per_decade - CK.sen_mine_per_decade).abs().max())
print('max |p difference|  (original) :', (CK.p_pkg_orig - CK.p_mine_orig).abs().max())
CK.round(5)

max |p difference|  (Hamed-Rao): 2.220446049250313e-16
max |Z difference|  (Hamed-Rao): 8.881784197001252e-16
max |Sen slope difference|     : 0.0
max |p difference|  (original) : 0.0


,Location,Index,p_pkg_HR,p_mine_HR,z_pkg_HR,z_mine_HR,sen_pkg_per_decade,sen_mine_per_decade,p_pkg_orig,p_mine_orig
0,Abuja,SPI-3,0.00000,0.00000,-4.89731,-4.89731,-0.34139,-0.34139,0.00000,0.00000
1,Gombe,SPI-3,0.77156,0.77156,-0.29034,-0.29034,-0.02627,-0.02627,0.42531,0.42531
2,Ibadan,SPI-3,0.00430,0.00430,2.85513,2.85513,0.16158,0.16158,0.00009,0.00009
3,Ikot Akpaden,SPI-3,0.85982,0.85982,-0.17660,-0.17660,-0.01878,-0.01878,0.65166,0.65166
4,Jalingo,SPI-3,0.00016,0.00016,-3.78262,-3.78262,-0.24655,-0.24655,0.00000,0.00000
5,Kukawa,SPI-3,0.05617,0.05617,1.90968,1.90968,0.15527,0.15527,0.00140,0.00140
6,Sokoto,SPI-3,0.00000,0.00000,4.82824,4.82824,0.18797,0.18797,0.00000,0.00000
7,Yenagoa,SPI-3,0.24148,0.24148,-1.17130,-1.17130,-0.14475,-0.14475,0.00062,0.00062
8,Abuja,SPEI-3,0.00000,0.00000,-4.72388,-4.72388,-0.46398,-0.46398,0.00000,0.00000
9,Gombe,SPEI-3,0.32819,0.32819,-0.97776,-0.97776,-0.12009,-0.12009,0.00294,0.00294


## Analytic ITA significance limits (Sen 2017) - valid for independent data only

In [19]:
def sen2017_ita(x1, x2, alpha=0.05):
    """Sen (2017) ITA slope and analytic confidence limit (formula as supplied; please confirm against the paper).
        s        = 2 (mean(sorted x2) - mean(sorted x1)) / n          (n = length of the whole series)
        var(s)   = 8 sigma^2 (1 - rho12) / n^2                        (sigma^2 = variance of the whole series,
                                                                       rho12 = correlation of the two SORTED halves)
        limit    = z_(1-alpha/2) * sd(s);  significant if |s| > limit
    Assumes independent observations: it over-rejects for autocorrelated series."""
    x1, x2 = np.asarray(x1, float), np.asarray(x2, float); n = len(x1) + len(x2)
    s = 2 * (x2.mean() - x1.mean()) / n
    rho12 = np.corrcoef(np.sort(x1), np.sort(x2))[0, 1]
    sd_s = np.sqrt(8.0 * np.var(np.concatenate([x1, x2]), ddof=1) * max(1 - rho12, 1e-12)) / n
    lim = stats.norm.ppf(1 - alpha / 2) * sd_s
    return s, lim, abs(s) > lim, rho12

def alashan2020_ita(x1, x2, alpha=0.05):
    """Alashan (2020) ITA test (procedure as supplied; please confirm against the paper).
        s        = 2 (mean(sorted x2) - mean(sorted x1)) / n
        sigma_s  = (2/n) * sqrt(s1^2 + s2^2 - 2 rho12 s1 s2)    s1, s2 = std of the sorted halves, rho12 = their correlation
        Z_ITA    = s / sigma_s ;  significant if |Z_ITA| > z_(1-alpha/2)
    Assumes independent observations."""
    a, b = np.sort(np.asarray(x1, float)), np.sort(np.asarray(x2, float)); n = len(a) + len(b)
    s = 2 * (b.mean() - a.mean()) / n
    s1, s2 = a.std(ddof=1), b.std(ddof=1); rho = np.corrcoef(a, b)[0, 1]
    sd = (2.0 / n) * np.sqrt(max(s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2, 1e-12))
    z = s / sd
    return s, z, abs(z) > stats.norm.ppf(1 - alpha / 2)

def boot_meandiff_sig(x1, x2, L, B=2000, seed=0):
    rng = np.random.default_rng(seed); d = [cbb(x2, L, rng).mean() - cbb(x1, L, rng).mean() for _ in range(B)]
    lo, hi = np.percentile(d, [2.5, 97.5]); return lo > 0 or hi < 0

out = []
for sc in (3, 12):
    L = 12 if sc == 3 else 24
    for kind in ('SPI', 'SPEI'):
        for loc in P.columns:
            s = series_for(loc, kind, sc); x1, x2 = s.loc[X_START:X_END].values, s.loc[Y_START:Y_END].values
            sl, lim, sig, rho = sen2017_ita(x1, x2)
            _, zal, sig_al = alashan2020_ita(x1, x2)
            out.append(dict(Location=loc.title(), Index=f'{kind}-{sc}', ITA_slope_per_decade=sl * 120,
                            slope_limit_per_decade=lim * 120, rho12_sorted=round(rho, 4), Sen2017_sig=sig, Alashan_Z=zal, Alashan2020_sig=sig_al,
                            bootstrap_sig=boot_meandiff_sig(x1, x2, L), MMK_sig=mk_test(s.values, hr=True)['p'] < 0.05))
AN = pd.DataFrame(out)
print('significant out of 32:  Sen2017 analytic:', int(AN.Sen2017_sig.sum()), '| Alashan2020:', int(AN.Alashan2020_sig.sum()), '| block bootstrap:', int(AN.bootstrap_sig.sum()),
      '| modified MK:', int(AN.MMK_sig.sum()))
AN.round(4)

significant out of 32:  Sen2017 analytic: 21 | Alashan2020: 19 | block bootstrap: 16 | modified MK: 13


,Location,Index,ITA_slope_per_decade,slope_limit_per_decade,rho12_sorted,Sen2017_sig,Alashan_Z,Alashan2020_sig,bootstrap_sig,MMK_sig
0,Abuja,SPI-3,-0.3576,0.1520,0.9878,True,-4.4543,True,True,True
1,Gombe,SPI-3,-0.0733,0.2177,0.9684,False,-0.3858,False,False,False
2,Ibadan,SPI-3,0.1231,0.1328,0.9908,False,1.7839,False,False,True
3,Ikot Akpaden,SPI-3,-0.0519,0.1560,0.9874,False,-0.2856,False,False,False
4,Jalingo,SPI-3,-0.3018,0.1519,0.9875,True,-3.5069,True,True,True
5,Kukawa,SPI-3,0.0837,0.1486,0.9913,False,1.0802,False,False,False
6,Sokoto,SPI-3,0.2113,0.0993,0.9939,True,2.2530,True,True,True
7,Yenagoa,SPI-3,-0.2083,0.1252,0.9918,True,-1.8499,False,True,False
8,Abuja,SPEI-3,-0.5438,0.2014,0.9780,True,-4.0468,True,True,True
9,Gombe,SPEI-3,-0.2798,0.1683,0.9839,True,-1.9888,True,True,False


## Export

In [20]:
with pd.ExcelWriter(os.path.join(OUT, 'crosscheck_results.xlsx'), engine='openpyxl') as w:
    CK.to_excel(w, sheet_name='pymannkendall vs mine', index=False)
    AN.to_excel(w, sheet_name='ITA analytic limits', index=False)
print('saved to', OUT)

saved to sh_ita_outputs
